# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from lightgbm import LGBMClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [4]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.868510,0.826590
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.772514,0.801335
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.717012,0.782757
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,0.005494,0.502476
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,0.482416,0.703165


In [5]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,0.019849,0.509303
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,0.012173,0.505573
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,0.010304,0.504827
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.444558,0.696136
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.939648,0.852833


# Machine Learning

In [6]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "objective": "binary",
        "n_estimators": trial.suggest_int("n_estimators", 30, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 4),
        "num_leaves": trial.suggest_int("num_leaves", 2, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 100, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 100, log=True),
        "verbosity": -1,
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-18 15:15:17,166] A new study created in memory with name: no-name-53b3bd95-c4b7-4452-bc70-b4fe1d1446f7
Best trial: 6. Best value: 0.947241:   0%|          | 1/500 [00:11<1:34:36, 11.38s/it]

[I 2026-05-18 15:15:28,513] Trial 6 finished with value: 0.9472414821263916 and parameters: {'n_estimators': 38, 'learning_rate': 0.009708146897133584, 'max_depth': 4, 'num_leaves': 9, 'min_child_samples': 114, 'subsample': 0.6042135771372108, 'colsample_bytree': 0.9848710065874089, 'reg_alpha': 96.33955231721637, 'reg_lambda': 0.0002154455811387925}. Best is trial 6 with value: 0.9472414821263916.


Best trial: 7. Best value: 0.947997:   0%|          | 2/500 [00:20<1:22:36,  9.95s/it]

[I 2026-05-18 15:15:37,460] Trial 7 finished with value: 0.9479972785766202 and parameters: {'n_estimators': 187, 'learning_rate': 0.017626056609156185, 'max_depth': 1, 'num_leaves': 12, 'min_child_samples': 237, 'subsample': 0.7534907220707546, 'colsample_bytree': 0.7705308328305387, 'reg_alpha': 0.05564883951719721, 'reg_lambda': 1.2143407865983868e-05}. Best is trial 7 with value: 0.9479972785766202.


Best trial: 11. Best value: 0.948832:   1%|          | 3/500 [00:23<57:30,  6.94s/it]  

...

[I 2026-05-18 15:49:21,293] Trial 498 finished with value: 0.9497895807978745 and parameters: {'n_estimators': 297, 'learning_rate': 0.014688927876693594, 'max_depth': 4, 'num_leaves': 9, 'min_child_samples': 158, 'subsample': 0.6977283860805269, 'colsample_bytree': 0.6924517965632129, 'reg_alpha': 2.235968660574188, 'reg_lambda': 1.1056580041163073e-05}. Best is trial 343 with value: 0.9499011058715915.


Best trial: 343. Best value: 0.949901: 100%|█████████▉| 499/500 [34:04<00:01,  1.81s/it]

[I 2026-05-18 15:49:21,568] Trial 496 finished with value: 0.9498930055455018 and parameters: {'n_estimators': 298, 'learning_rate': 0.042255400622819426, 'max_depth': 4, 'num_leaves': 13, 'min_child_samples': 156, 'subsample': 0.6990952524460817, 'colsample_bytree': 0.6846759010680863, 'reg_alpha': 2.417725671012694, 'reg_lambda': 1.0108354030745652e-05}. Best is trial 343 with value: 0.9499011058715915.
[I 2026-05-18 15:49:21,758] Trial 499 finished with value: 0.9497926349546393 and parameters: {'n_estimators': 297, 'learning_rate': 0.015564554998086434, 'max_depth': 4, 'num_leaves': 9, 'min_child_samples': 156, 'subsample': 0.7054783895971125, 'colsample_bytree': 0.6829791075200691, 'reg_alpha': 2.4958000525311514, 'reg_lambda': 1.0552355486939335e-05}. Best is trial 343 with value: 0.9499011058715915.


Best trial: 343. Best value: 0.949901: 100%|██████████| 500/500 [34:04<00:00,  4.09s/it]

[I 2026-05-18 15:49:22,153] Trial 497 finished with value: 0.949718719754979 and parameters: {'n_estimators': 296, 'learning_rate': 0.008666992162793518, 'max_depth': 4, 'num_leaves': 13, 'min_child_samples': 157, 'subsample': 0.7117558616291776, 'colsample_bytree': 0.6939519686598397, 'reg_alpha': 2.1535669572688754, 'reg_lambda': 1.0333675731637723e-05}. Best is trial 343 with value: 0.9499011058715915.
Best trial score:
0.9499011058715915

Best params:
{'n_estimators': 291, 'learning_rate': 0.04680748707140611, 'max_depth': 4, 'num_leaves': 15, 'min_child_samples': 108, 'subsample': 0.6139365796661329, 'colsample_bytree': 0.6315798031368468, 'reg_alpha': 5.8966657985979225, 'reg_lambda': 1.0109532216966318e-05}


In [7]:
stacking_lgbm = LGBMClassifier(**study.best_params).fit(X_train, y_train.PitNextLap)

[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001287 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

# Submission

In [8]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [11]:
submission['PitNextLap'] = stacking_lgbm.predict_proba(X_test)[:, 1]

In [12]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [13]:
X_train.columns

Index(['lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba',
       'knn_proba', 'voting_lgbm_cat_xgb_hist_proba',
       'stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba'],
      dtype='str')